# BG-NBD ve Gamma-Gamma ile CLTV Tahmini (FLO)

FLO OmniChannel müşteri verisi üzerinde BG/NBD ve Gamma-Gamma modelleri ile müşteri yaşam boyu değeri (CLTV) tahmini.

## İş Problemi (Business Problem)

FLO satış ve pazarlama faaliyetleri için roadmap belirlemek istemektedir. Şirketin orta uzun vadeli plan yapabilmesi için var olan müşterilerin gelecekte şirkete sağlayacakları potansiyel değerin tahmin edilmesi gerekmektedir.

## Veri Seti Hikayesi

Veri seti, son alışverişlerini 2020 - 2021 yıllarında OmniChannel (hem online hem offline alışveriş yapan) olarak yapan müşterilerin geçmiş alışveriş davranışlarından elde edilen bilgilerden oluşmaktadır.

**Değişkenler**

- **master_id:** Eşsiz müşteri numarası
- **order_channel:** Alışveriş yapılan platforma ait hangi kanalın kullanıldığı (Android, ios, Desktop, Mobile, Offline)
- **last_order_channel:** En son alışverişin yapıldığı kanal
- **first_order_date:** Müşterinin yaptığı ilk alışveriş tarihi
- **last_order_date:** Müşterinin yaptığı son alışveriş tarihi
- **last_order_date_online:** Müşterinin online platformda yaptığı son alışveriş tarihi
- **last_order_date_offline:** Müşterinin offline platformda yaptığı son alışveriş tarihi
- **order_num_total_ever_online:** Müşterinin online platformda yaptığı toplam alışveriş sayısı
- **order_num_total_ever_offline:** Müşterinin offline'da yaptığı toplam alışveriş sayısı
- **customer_value_total_ever_offline:** Müşterinin offline alışverişlerinde ödediği toplam ücret
- **customer_value_total_ever_online:** Müşterinin online alışverişlerinde ödediği toplam ücret
- **interested_in_categories_12:** Müşterinin son 12 ayda alışveriş yaptığı kategorilerin listesi

## Görevler

1. **GÖREV 1:** Veriyi Hazırlama
2. **GÖREV 2:** CLTV Veri Yapısının Oluşturulması
3. **GÖREV 3:** BG/NBD, Gamma-Gamma Modellerinin Kurulması, CLTV'nin Hesaplanması
4. **GÖREV 4:** CLTV'ye Göre Segmentlerin Oluşturulması
5. **BONUS:** Tüm süreci fonksiyonlaştırınız.

In [1]:
import datetime as dt
import pandas as pd
import numpy as np
from lifetimes import BetaGeoFitter
from lifetimes import GammaGammaFitter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

---
## GÖREV 1: Veriyi Hazırlama

### 1. Veriyi Okuma

`flo_data_20k.csv` verisini okuyunuz. DataFrame'in kopyasını oluşturunuz.

In [2]:
df = pd.read_csv("datasets/flo_data_20k.csv")
df_copy = df.copy()

### 2. Aykırı Değer Fonksiyonları

Aykırı değerleri baskılamak için `outlier_thresholds` ve `replace_with_thresholds` fonksiyonlarını tanımlayınız.

> **Not:** CLTV hesaplanırken frequency değerleri integer olması gerekmektedir. Bu nedenle alt ve üst limitlerini `round()` ile yuvarlayınız.

In [3]:
def outlier_thresholds(dataframe, variable):
    quartile1 = dataframe[variable].quantile(0.01)
    quartile3 = dataframe[variable].quantile(0.99)
    interquantile_range = quartile3 - quartile1
    up_limit = round(quartile3 + 1.5 * interquantile_range)
    low_limit = round(quartile1 - 1.5 * interquantile_range)
    return low_limit, up_limit


def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

### 3. Aykırı Değerleri Baskılama

`order_num_total_ever_online`, `order_num_total_ever_offline`, `customer_value_total_ever_offline`, `customer_value_total_ever_online` değişkenlerinin aykırı değerleri varsa baskılayınız.

In [4]:
outlier_cols = [
    "order_num_total_ever_online",
    "order_num_total_ever_offline",
    "customer_value_total_ever_offline",
    "customer_value_total_ever_online",
]

for col in outlier_cols:
    replace_with_thresholds(df_copy, col)

df_copy[outlier_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
order_num_total_ever_online,19945.000,3.092,3.810,1.000,1.000,2.000,4.000,48.000
order_num_total_ever_offline,19945.000,1.886,1.435,1.000,1.000,1.000,2.000,16.000
customer_value_total_ever_offline,19945.000,251.921,251.024,10.000,99.990,179.980,319.970,3020.000
customer_value_total_ever_online,19945.000,489.706,632.610,12.990,149.980,286.460,578.440,7800.000


### 4. Omnichannel Toplam Değişkenleri

Omnichannel müşterilerin hem online'dan hem de offline platformlardan alışveriş yaptığını ifade etmektedir. Her bir müşterinin toplam alışveriş sayısı ve harcaması için yeni değişkenler oluşturun.

In [5]:
df_copy["order_num_total"] = (
    df_copy["order_num_total_ever_online"] + df_copy["order_num_total_ever_offline"]
)
df_copy["customer_value_total"] = (
    df_copy["customer_value_total_ever_offline"] + df_copy["customer_value_total_ever_online"]
)

df_copy[["order_num_total", "customer_value_total"]].head()

,order_num_total,customer_value_total
0,5.000,939.370
1,21.000,2013.550
2,5.000,585.320
3,2.000,121.970
4,2.000,209.980


### 5. Tarih Değişkenlerinin Dönüştürülmesi

Değişken tiplerini inceleyiniz. Tarih ifade eden değişkenlerin tipini `date`'e çeviriniz.

In [6]:
date_cols = [
    "first_order_date",
    "last_order_date",
    "last_order_date_online",
    "last_order_date_offline",
]

df_copy[date_cols] = df_copy[date_cols].apply(pd.to_datetime)
df_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 19945 entries, 0 to 19944
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   master_id                          19945 non-null  str           
 1   order_channel                      19945 non-null  str           
 2   last_order_channel                 19945 non-null  str           
 3   first_order_date                   19945 non-null  datetime64[us]
 4   last_order_date                    19945 non-null  datetime64[us]
 5   last_order_date_online             19945 non-null  datetime64[us]
 6   last_order_date_offline            19945 non-null  datetime64[us]
 7   order_num_total_ever_online        19945 non-null  float64       
 8   order_num_total_ever_offline       19945 non-null  float64       
 9   customer_value_total_ever_offline  19945 non-null  float64       
 10  customer_value_total_ever_online   19945 non-

---
## GÖREV 2: CLTV Veri Yapısının Oluşturulması

### 1. Analiz Tarihinin Belirlenmesi

Veri setindeki en son alışverişin yapıldığı tarihten 2 gün sonrasını analiz tarihi olarak alınız.

In [7]:
today_date = df_copy["last_order_date"].max() + pd.Timedelta(days=2)
today_date

Timestamp('2021-06-01 00:00:00')

### 2. CLTV DataFrame'inin Oluşturulması

`customer_id`, `recency_cltv_weekly`, `T_weekly`, `frequency` ve `monetary_cltv_avg` değerlerinin yer aldığı yeni bir `cltv_df` oluşturunuz.

- **Monetary:** Satın alma başına ortalama değer
- **Recency ve Tenure (T):** Haftalık cinsten ifade edilecek

In [ ]:
# Burada CLTV (Customer Lifetime Value) hesaplamasında kullanılacak temel veri çerçevesi oluşturuluyor. Açıklamalarıyla birlikte adım adım gidelim:

# 1. "customer_id": Her satırda müşteri özelinde CLTV hesaplanacak. Burada "master_id", müşteri ID'sidir.
# 2. "recency_cltv_weekly": Müşterinin ilk alışverişi ile son alışverişi arasında geçen süreyi haftalık olarak hesaplar. Yani, müşteri kaç hafta boyunca alışveriş yapmış.
# 3. "T_weekly": Analiz tarihine kadar, müşterinin ilk alışverişinden itibaren geçen toplam haftayı ifade eder.
# 4. "frequency": Müşterinin yaptığı toplam alışveriş sayısını belirtir.
# 5. "monetary_cltv_avg": Müşterinin alışveriş başına ortalama ne kadar harcama yaptığını gösterir.

cltv_df = pd.DataFrame({
    "customer_id": df_copy["master_id"],  # Müşteri kimliği
    "recency_cltv_weekly": (df_copy["last_order_date"] - df_copy["first_order_date"]).dt.days / 7,  # Son ve ilk alışveriş arası geçen hafta
    "T_weekly": (today_date - df_copy["first_order_date"]).dt.days / 7,  # İlk alışverişten analiz tarihine kadar geçen hafta
    "frequency": df_copy["order_num_total"],  # Toplam alışveriş adedi
    "monetary_cltv_avg": df_copy["customer_value_total"] / df_copy["order_num_total"],  # Ortalama alışveriş başına harcama
})

cltv_df.head()  # Oluşan CLTV veri çerçevesinin ilk 5 satırını gösterir.

,customer_id,recency_cltv_weekly,T_weekly,frequency,monetary_cltv_avg
0,cc294636-19f0-11eb-8d74-000d3a38a36f,17.000,30.571,5.000,187.874
1,f431bd5a-ab7b-11e9-a2fc-000d3a38a36f,209.857,224.857,21.000,95.883
2,69b69676-1a40-11ea-941b-000d3a38a36f,52.286,78.857,5.000,117.064
3,1854e56c-491f-11eb-806e-000d3a38a36f,1.571,20.857,2.000,60.985
4,d6ea1074-f1f5-11e9-9346-000d3a38a36f,83.143,95.429,2.000,104.990


---
## GÖREV 3: BG/NBD, Gamma-Gamma Modellerinin Kurulması, 6 Aylık CLTV'nin Hesaplanması

### 1. BG/NBD Modelinin Kurulması

BG/NBD modelini fit ediniz.

In [10]:
bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(
    cltv_df["frequency"],
    cltv_df["recency_cltv_weekly"],
    cltv_df["T_weekly"],
)
bgf

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


<lifetimes.BetaGeoFitter: fitted with 19945 subjects, a: 0.00, alpha: 76.17, b: 0.00, r: 3.66>

### 2. Beklenen Satın Alma Tahminleri

- 3 ay içerisinde müşterilerden beklenen satın almaları tahmin ediniz ve `exp_sales_3_month` olarak `cltv_df`'e ekleyiniz.
- 6 ay içerisinde müşterilerden beklenen satın almaları tahmin ediniz ve `exp_sales_6_month` olarak `cltv_df`'e ekleyiniz.
- 3. ve 6. aydaki en çok satın alım gerçekleştirecek 10 kişiyi inceleyiniz.

In [11]:
cltv_df["exp_sales_3_month"] = bgf.predict(
    4 * 3,
    cltv_df["frequency"],
    cltv_df["recency_cltv_weekly"],
    cltv_df["T_weekly"],
)

cltv_df["exp_sales_6_month"] = bgf.predict(
    4 * 6,
    cltv_df["frequency"],
    cltv_df["recency_cltv_weekly"],
    cltv_df["T_weekly"],
)

# 3 aylık en çok satın alım gerçekleştirecek 10 müşteri
cltv_df.sort_values("exp_sales_3_month", ascending=False).head(10)

,customer_id,recency_cltv_weekly,T_weekly,frequency,monetary_cltv_avg,exp_sales_3_month,exp_sales_6_month
7330,a4d534a2-5b1b-11eb-8dbd-000d3a38a36f,62.714,67.286,52.000,166.225,4.656,9.312
15611,4a7e875e-e6ce-11ea-8f44-000d3a38a36f,39.714,40.000,29.000,165.298,3.374,6.748
8328,1902bf80-0035-11eb-8341-000d3a38a36f,28.857,33.286,25.000,97.440,3.142,6.285
19538,55d54d9e-8ac7-11ea-8ec0-000d3a38a36f,52.571,58.714,31.000,228.530,3.084,6.168
14373,f00ad516-c4f4-11ea-98f7-000d3a38a36f,38.000,46.429,27.000,141.355,3.001,6.003
10489,7af5cd16-b100-11e9-9757-000d3a38a36f,103.143,111.857,43.000,157.113,2.978,5.956
4315,d5ef8058-a5c6-11e9-a2fc-000d3a38a36f,133.143,147.143,49.000,161.847,2.830,5.660
6756,27310582-6362-11ea-a6dc-000d3a38a36f,62.714,64.143,29.000,168.881,2.793,5.587
6666,53fe00d4-7b7a-11eb-960b-000d3a38a36f,9.714,13.000,17.000,259.865,2.781,5.561
10536,e143b6fa-d6f8-11e9-93bc-000d3a38a36f,104.571,113.429,40.000,176.200,2.763,5.527


In [12]:
cltv_df.sort_values("exp_sales_6_month", ascending=False).head(10)

,customer_id,recency_cltv_weekly,T_weekly,frequency,monetary_cltv_avg,exp_sales_3_month,exp_sales_6_month
7330,a4d534a2-5b1b-11eb-8dbd-000d3a38a36f,62.714,67.286,52.000,166.225,4.656,9.312
15611,4a7e875e-e6ce-11ea-8f44-000d3a38a36f,39.714,40.000,29.000,165.298,3.374,6.748
8328,1902bf80-0035-11eb-8341-000d3a38a36f,28.857,33.286,25.000,97.440,3.142,6.285
19538,55d54d9e-8ac7-11ea-8ec0-000d3a38a36f,52.571,58.714,31.000,228.530,3.084,6.168
14373,f00ad516-c4f4-11ea-98f7-000d3a38a36f,38.000,46.429,27.000,141.355,3.001,6.003
10489,7af5cd16-b100-11e9-9757-000d3a38a36f,103.143,111.857,43.000,157.113,2.978,5.956
4315,d5ef8058-a5c6-11e9-a2fc-000d3a38a36f,133.143,147.143,49.000,161.847,2.830,5.660
6756,27310582-6362-11ea-a6dc-000d3a38a36f,62.714,64.143,29.000,168.881,2.793,5.587
6666,53fe00d4-7b7a-11eb-960b-000d3a38a36f,9.714,13.000,17.000,259.865,2.781,5.561
10536,e143b6fa-d6f8-11e9-93bc-000d3a38a36f,104.571,113.429,40.000,176.200,2.763,5.527


### 3. Gamma-Gamma Modelinin Kurulması

Gamma-Gamma modelini fit ediniz. Müşterilerin ortalama bırakacakları değeri tahminleyip `exp_average_value` olarak `cltv_df`'e ekleyiniz.

In [13]:
# Gamma-Gamma modeli, müşterilerin işlem başına ortalama getirdiği parasal değeri tahmin etmek için kullanılmaktadır.
# Modelin kurulumu için GammaGammaFitter sınıfı kullanılır ve penalizer_coef hiperparametresi ile aşırı uyum önlenir.

ggf = GammaGammaFitter(penalizer_coef=0.01)  # Model nesnesi oluşturuluyor.
ggf.fit(cltv_df["frequency"], cltv_df["monetary_cltv_avg"])  # Model; müşterilerin işlem sıklığı ve işlem başına ortalama tutarı ile eğitiliyor.

# Model eğitildikten sonra her müşteri için beklenen ortalama kâr (exp_average_value) tahmini yapılır ve veri setine eklenir.
cltv_df["exp_average_value"] = ggf.conditional_expected_average_profit(
    cltv_df["frequency"],
    cltv_df["monetary_cltv_avg"],
)

# exp_average_value değerine göre en yüksek 10 müşteri sıralanır.
cltv_df.sort_values("exp_average_value", ascending=False).head(10)

,customer_id,recency_cltv_weekly,T_weekly,frequency,monetary_cltv_avg,exp_sales_3_month,exp_sales_6_month,exp_average_value
9055,47a642fe-975b-11eb-8c2a-000d3a38a36f,2.857,7.857,4.000,1401.800,1.094,2.189,1449.060
17323,f59053e2-a503-11e9-a2fc-000d3a38a36f,51.714,101.000,7.000,1106.467,0.722,1.444,1127.612
15516,9083981a-f59e-11e9-841e-000d3a38a36f,63.571,83.857,4.000,1090.360,0.575,1.149,1127.354
6402,851de3b4-8f0c-11eb-8cb8-000d3a38a36f,8.286,9.429,2.000,862.690,0.794,1.588,923.680
16410,6fecd6c8-261a-11ea-8e1c-000d3a38a36f,57.000,94.857,2.000,859.580,0.397,0.795,920.358
1853,f02473b0-43c3-11eb-806e-000d3a38a36f,17.286,23.143,2.000,835.875,0.684,1.369,895.037
7936,ae4ce104-dbd4-11ea-8757-000d3a38a36f,3.714,42.000,3.000,844.347,0.677,1.353,883.288
9738,3a27b334-dff4-11ea-acaa-000d3a38a36f,40.000,41.143,3.000,837.057,0.682,1.363,875.674
12828,0c24fc44-2ac8-11ea-9d27-000d3a38a36f,68.000,84.286,2.000,779.265,0.424,0.847,834.568
2291,26ac1432-1dd3-11ea-8bf2-000d3a38a36f,55.714,97.714,3.000,780.557,0.460,0.920,816.663


### 4. 6 Aylık CLTV Hesaplanması

6 aylık CLTV hesaplayınız ve `cltv` ismiyle dataframe'e ekleyiniz. CLTV değeri en yüksek 20 kişiyi gözlemleyiniz.

In [14]:
# Gamma-Gamma ve BG/NBD modelleri kullanılarak 6 aylık müşteri yaşam boyu değeri (CLTV) hesaplanmaktadır.
# customer_lifetime_value fonksiyonu, aşağıdaki parametrelerle uygulanır:
#   - bgf: Satın alma sıklığını tahmin eden BG/NBD modeli,
#   - frequency: Müşterinin alışveriş sıklığı,
#   - recency_cltv_weekly: Müşterinin ilk ve son satın alım arasındaki süre (hafta),
#   - T_weekly: Müşterinin ilk satın almasından analiz tarihine kadar geçen süre (hafta),
#   - monetary_cltv_avg: Satın alma başına ortalama harcama,
#   - time: CLTV'nin hesaplanacağı süre (6 ay = 6 hafta parametresi, çünkü freq="W" olarak ayarlandı),
#   - freq: Zaman periyodu, burada "W" yani haftalık,
#   - discount_rate: Iskontolu nakit akışı için diskont oranı (0.01 = %1).

cltv = ggf.customer_lifetime_value(
    bgf,
    cltv_df["frequency"],
    cltv_df["recency_cltv_weekly"],
    cltv_df["T_weekly"],
    cltv_df["monetary_cltv_avg"],
    time=6,                # 6 aylık CLTV için (freq="W" olduğundan 6 hafta)
    freq="W",              # Haftalık zaman periyodu kullanılıyor
    discount_rate=0.01,    # %1 indirim oranı (diskont)
)

# Hesaplanan CLTV değerleri dataframe'e yeni bir sütun olarak ekleniyor.
cltv_df["cltv"] = cltv

# CLTV değerine göre en yüksek 20 müşteri ekrana getiriliyor.
cltv_df.sort_values("cltv", ascending=False).head(20)

,customer_id,recency_cltv_weekly,T_weekly,frequency,monetary_cltv_avg,exp_sales_3_month,exp_sales_6_month,exp_average_value,cltv
9055,47a642fe-975b-11eb-8c2a-000d3a38a36f,2.857,7.857,4.000,1401.800,1.094,2.189,1449.060,3327.777
13880,7137a5c0-7aad-11ea-8f20-000d3a38a36f,6.143,13.143,11.000,758.085,1.970,3.940,767.361,3172.394
17323,f59053e2-a503-11e9-a2fc-000d3a38a36f,51.714,101.000,7.000,1106.467,0.722,1.444,1127.612,1708.982
12438,625f40a2-5bd2-11ea-98b0-000d3a38a36f,74.286,74.571,16.000,501.874,1.565,3.131,506.167,1662.613
7330,a4d534a2-5b1b-11eb-8dbd-000d3a38a36f,62.714,67.286,52.000,166.225,4.656,9.312,166.712,1628.887
8868,9ce6e520-89b0-11ea-a6e7-000d3a38a36f,3.429,34.429,8.000,601.226,1.265,2.531,611.493,1623.813
6402,851de3b4-8f0c-11eb-8cb8-000d3a38a36f,8.286,9.429,2.000,862.690,0.794,1.588,923.680,1538.856
6666,53fe00d4-7b7a-11eb-960b-000d3a38a36f,9.714,13.000,17.000,259.865,2.781,5.561,262.073,1529.228
19538,55d54d9e-8ac7-11ea-8ec0-000d3a38a36f,52.571,58.714,31.000,228.530,3.084,6.168,229.607,1485.819
14858,031b2954-6d28-11eb-99c4-000d3a38a36f,14.857,15.571,3.000,743.587,0.872,1.743,778.050,1423.000


---
## GÖREV 4: CLTV'ye Göre Segmentlerin Oluşturulması

### 1. CLTV Segmentasyonu

6 aylık CLTV'ye göre tüm müşterilerinizi 4 gruba (segmente) ayırınız ve grup isimlerini `cltv_segment` ismi ile dataframe'e ekleyiniz.

In [ ]:
# 6 aylık CLTV değerlerine göre tüm müşteriler 4 segmente ayrılıyor.
# pd.qcut fonksiyonu, 'cltv' sütununu dört eşit gruba (çeyrekliklere) böler ve her kümeye bir etiket atar:
# - "A": En yüksek CLTV değerine sahip müşteriler,
# - "B": İkinci en yüksek grup,
# - "C": Daha düşük CLTV'ye sahip grup,
# - "D": En düşük CLTV'li müşteriler.
# Böylece segmentasyon A (çok iyi) > B > C > D (düşük) şeklindedir.

cltv_df["cltv_segment"] = pd.qcut(
    cltv_df["cltv"], 
    4, 
    labels=["D", "C", "B", "A"]
)

# Bu işlem sonucunda her müşteri 'cltv_segment' sütununda ait olduğu segmenti görmüş olur.
cltv_df.head()

,customer_id,recency_cltv_weekly,T_weekly,frequency,monetary_cltv_avg,exp_sales_3_month,exp_sales_6_month,exp_average_value,cltv,cltv_segment
0,cc294636-19f0-11eb-8d74-000d3a38a36f,17.000,30.571,5.000,187.874,0.974,1.948,193.633,395.733,A
1,f431bd5a-ab7b-11e9-a2fc-000d3a38a36f,209.857,224.857,21.000,95.883,0.983,1.966,96.665,199.431,B
2,69b69676-1a40-11ea-941b-000d3a38a36f,52.286,78.857,5.000,117.064,0.671,1.341,120.968,170.224,B
3,1854e56c-491f-11eb-806e-000d3a38a36f,1.571,20.857,2.000,60.985,0.700,1.401,67.320,98.946,D
4,d6ea1074-f1f5-11e9-9346-000d3a38a36f,83.143,95.429,2.000,104.990,0.396,0.792,114.325,95.012,D


### 2. Segment Ortalamalarının İncelenmesi

Segmentlerin recency, frequency ve monetary ortalamalarını inceleyiniz.

In [16]:
# Müşteriler cltv_segment sütununa göre gruplandırılıyor ve her segmentin recency (hafta), frequency ve ortalama harcama (monetary) değerlerinin ortalaması alınıyor.
# Böylece her segmentin müşteri davranış özellikleri daha kolay karşılaştırılabiliyor.
segment_ortalamalari = (
    cltv_df
    .groupby("cltv_segment")[
        ["recency_cltv_weekly", "frequency", "monetary_cltv_avg"]
    ]
    .mean()
    .round(2)
)

# Elde edilen tablo her segmentin ortalama alışveriş sıklığı (frequency), ortalama harcama (monetary) ve son alışverişten bu yana geçen haftayı (recency) gösterir.
segment_ortalamalari

,recency_cltv_weekly,frequency,monetary_cltv_avg
cltv_segment,,,
D,139.000,3.770,93.150
C,92.630,4.400,125.790
B,81.990,5.090,160.640
A,67.430,6.650,228.830


### 3. Aksiyon Önerileri

4 grup içerisinden seçeceğiniz 2 grup için yönetime kısa kısa 6 aylık aksiyon önerilerinde bulununuz.

**Örnek öneriler:**

- **A Segmenti (En Yüksek CLTV):** VIP müşteri programı, erken erişim kampanyaları ve kişiselleştirilmiş öneriler ile elde tutma odaklı strateji.
- **D Segmenti (En Düşük CLTV):** Düşük maliyetli yeniden aktivasyon kampanyaları ve çapraz satış fırsatları ile müşteriyi tekrar alışverişe çekme.

---
## BONUS: Tüm Süreci Fonksiyonlaştırma

Yukarıdaki tüm analiz sürecini tek bir fonksiyon veya pipeline halinde fonksiyonlaştırınız.

In [17]:
def create_cltv_flo(dataframe, month=6, discount_rate=0.01):
    df = dataframe.copy()

    outlier_cols = [
        "order_num_total_ever_online",
        "order_num_total_ever_offline",
        "customer_value_total_ever_offline",
        "customer_value_total_ever_online",
    ]
    for col in outlier_cols:
        replace_with_thresholds(df, col)

    df["order_num_total"] = (
        df["order_num_total_ever_online"] + df["order_num_total_ever_offline"]
    )
    df["customer_value_total"] = (
        df["customer_value_total_ever_offline"] + df["customer_value_total_ever_online"]
    )

    date_cols = [
        "first_order_date",
        "last_order_date",
        "last_order_date_online",
        "last_order_date_offline",
    ]
    df[date_cols] = df[date_cols].apply(pd.to_datetime)

    today_date = df["last_order_date"].max() + pd.Timedelta(days=2)

    cltv_df = pd.DataFrame({
        "customer_id": df["master_id"],
        "recency_cltv_weekly": (df["last_order_date"] - df["first_order_date"]).dt.days / 7,
        "T_weekly": (today_date - df["first_order_date"]).dt.days / 7,
        "frequency": df["order_num_total"],
        "monetary_cltv_avg": df["customer_value_total"] / df["order_num_total"],
    })

    bgf = BetaGeoFitter(penalizer_coef=0.001)
    bgf.fit(cltv_df["frequency"], cltv_df["recency_cltv_weekly"], cltv_df["T_weekly"])

    cltv_df["exp_sales_3_month"] = bgf.predict(
        4 * 3, cltv_df["frequency"], cltv_df["recency_cltv_weekly"], cltv_df["T_weekly"]
    )
    cltv_df["exp_sales_6_month"] = bgf.predict(
        4 * 6, cltv_df["frequency"], cltv_df["recency_cltv_weekly"], cltv_df["T_weekly"]
    )

    ggf = GammaGammaFitter(penalizer_coef=0.01)
    ggf.fit(cltv_df["frequency"], cltv_df["monetary_cltv_avg"])

    cltv_df["exp_average_value"] = ggf.conditional_expected_average_profit(
        cltv_df["frequency"], cltv_df["monetary_cltv_avg"]
    )

    cltv = ggf.customer_lifetime_value(
        bgf,
        cltv_df["frequency"],
        cltv_df["recency_cltv_weekly"],
        cltv_df["T_weekly"],
        cltv_df["monetary_cltv_avg"],
        time=month,
        freq="W",
        discount_rate=discount_rate,
    )
    cltv_df["cltv"] = cltv
    cltv_df["cltv_segment"] = pd.qcut(cltv_df["cltv"], 4, labels=["D", "C", "B", "A"])

    return cltv_df

In [18]:
cltv_final = create_cltv_flo(df)
cltv_final.head()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,customer_id,recency_cltv_weekly,T_weekly,frequency,monetary_cltv_avg,exp_sales_3_month,exp_sales_6_month,exp_average_value,cltv,cltv_segment
0,cc294636-19f0-11eb-8d74-000d3a38a36f,17.000,30.571,5.000,187.874,0.974,1.948,193.633,395.733,A
1,f431bd5a-ab7b-11e9-a2fc-000d3a38a36f,209.857,224.857,21.000,95.883,0.983,1.966,96.665,199.431,B
2,69b69676-1a40-11ea-941b-000d3a38a36f,52.286,78.857,5.000,117.064,0.671,1.341,120.968,170.224,B
3,1854e56c-491f-11eb-806e-000d3a38a36f,1.571,20.857,2.000,60.985,0.700,1.401,67.320,98.946,D
4,d6ea1074-f1f5-11e9-9346-000d3a38a36f,83.143,95.429,2.000,104.990,0.396,0.792,114.325,95.012,D


In [19]:
cltv_final.to_csv("flo_cltv_prediction.csv", index=False)